# 🐱 Cats vs Dogs — Neural Network Classifier
### PyTorch | Transfer Learning | Google Colab GPU

**Dataset:** Oxford-IIIT Pet Dataset (public, auto-downloaded)

**Model:** ResNet-18 with Transfer Learning (ImageNet pretrained)

**Backend Developer Agent** | Clean Architecture | GPU-Accelerated

---
> ⚠️ **Before running:** Set GPU runtime via `Runtime → Change Runtime Type → T4 GPU`

## 🔧 Section 1: GPU Check & Setup

In [ ]:
import torch

# ─── GPU Detection ───
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   CUDA version: {torch.version.cuda}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('✅ Apple MPS GPU detected.')
else:
    device = torch.device('cpu')
    print('⚠️  No GPU found — using CPU. Training will be slow.')
    print('   Go to Runtime → Change Runtime Type → GPU')

print(f'\nPyTorch version: {torch.__version__}')
print(f'Active device: {device}')

## 📦 Section 2: Install Dependencies

In [ ]:
# ─── Install required packages ───
# torch and torchvision are pre-installed in Colab; this ensures latest versions
!pip install -q torch torchvision tqdm matplotlib seaborn pillow scikit-learn
print('✅ Dependencies installed.')

## 📚 Section 3: Imports

In [ ]:
import os
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from typing import Tuple, List, Dict

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, random_split, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import OxfordIIITPet

from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('✅ All imports successful.')
print(f'   torchvision: {torchvision.__version__}')

## 🗂️ Section 4: Dataset — Oxford-IIIT Pets (Auto-Download)

**Oxford-IIIT Pet Dataset** is a public dataset with ~7,349 images of 37 pet breeds.
We use the `species` label to get binary Cat/Dog classification automatically.

| Label | Class | Species Code |
|---|---|---|
| 0 | Cat | 1 (in dataset) |
| 1 | Dog | 2 (in dataset) |

In [ ]:
# ─── Configuration ───
CONFIG = {
    'data_dir':     '/content/data',
    'output_dir':   '/content/output',
    'image_size':   224,
    'batch_size':   32,
    'epochs':       15,
    'lr':           1e-4,
    'val_split':    0.2,
    'num_workers':  2,
    'model_type':   'resnet18',  # 'custom_cnn', 'resnet18', 'resnet50'
}

os.makedirs(CONFIG['data_dir'],   exist_ok=True)
os.makedirs(CONFIG['output_dir'], exist_ok=True)

CLASS_NAMES = ['Cat', 'Dog']
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print('✅ Configuration:')
for k, v in CONFIG.items():
    print(f'   {k}: {v}')

In [ ]:
# ─── Data Transforms ───
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(CONFIG['image_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG['image_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

print('✅ Transforms defined.')

In [ ]:
# ─── Oxford Pets Binary Dataset Wrapper ───
class OxfordPetBinary(Dataset):
    """
    Wraps OxfordIIITPet, converts species to binary label.
    Species 1 (Cat) → 0.0, Species 2 (Dog) → 1.0
    """
    def __init__(self, root, split='trainval', transform=None, download=True):
        self.inner = OxfordIIITPet(
            root=root,
            split=split,
            target_types='species',
            transform=transform,
            download=download,
        )

    def __len__(self):
        return len(self.inner)

    def __getitem__(self, idx):
        image, species = self.inner[idx]
        # species: 1=Cat → 0, 2=Dog → 1
        label = torch.tensor(float(species - 1), dtype=torch.float32)
        return image, label


# ─── Download & Split ───
print('📥 Downloading Oxford-IIIT Pet Dataset (~750 MB)...')
print('   Source: https://www.robots.ox.ac.uk/~vgg/data/pets/')

full_train_ds = OxfordPetBinary(CONFIG['data_dir'], split='trainval', transform=train_transform, download=True)
full_val_ds   = OxfordPetBinary(CONFIG['data_dir'], split='trainval', transform=val_transform, download=False)

n_total = len(full_train_ds)
n_val   = int(n_total * CONFIG['val_split'])
n_train = n_total - n_val

gen = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(n_total), [n_train, n_val], generator=gen)

train_ds = Subset(full_train_ds, train_idx.indices)
val_ds   = Subset(full_val_ds,   val_idx.indices)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'], pin_memory=True)

print(f'\n✅ Dataset ready!')
print(f'   Total images : {n_total:,}')
print(f'   Training     : {n_train:,}')
print(f'   Validation   : {n_val:,}')
print(f'   Batches/epoch: {len(train_loader)}')

In [ ]:
# ─── Visualise Sample Images ───
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std  = torch.tensor(STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Sample Training Images — Cats & Dogs', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < 16:
        img   = denormalize(images[i]).permute(1, 2, 0).numpy()
        label = CLASS_NAMES[int(labels[i].item())]
        ax.imshow(img)
        ax.set_title(label, fontsize=9, color='green' if label=='Cat' else 'blue')
    ax.axis('off')

plt.tight_layout()
plt.show()
print(f'Batch shape: {images.shape}')

## 🧠 Section 5: Model Definition

In [ ]:
# ─── Custom CNN Architecture ───
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CatDogCNN(nn.Module):
    """Custom 4-block CNN for binary classification."""
    def __init__(self, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(128, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        return self.classifier(self.features(x))


# ─── Model Factory ───
def build_model(model_type='resnet18', pretrained=True):
    if model_type == 'custom_cnn':
        return CatDogCNN()
    elif model_type == 'resnet18':
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.resnet18(weights=weights)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
        return m
    elif model_type == 'resnet50':
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        m = models.resnet50(weights=weights)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
        return m
    else:
        raise ValueError(f'Unknown model: {model_type}')


# ─── Instantiate ───
model = build_model(CONFIG['model_type'], pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model: {CONFIG["model_type"]}')
print(f'   Trainable parameters: {n_params:,}')
print(f'   Running on: {device}')

## 🏋️ Section 6: Training Loop

In [ ]:
# ─── Loss, Optimizer, Scheduler ───
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)
scaler    = GradScaler(enabled=(device.type == 'cuda'))

best_val_acc = 0.0
best_ckpt    = os.path.join(CONFIG['output_dir'], 'best_model.pth')
history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('✅ Training setup complete.')
print(f'   Loss      : BCEWithLogitsLoss')
print(f'   Optimizer : AdamW (lr={CONFIG["lr"]})')
print(f'   Scheduler : CosineAnnealingLR')
print(f'   AMP       : {device.type == "cuda"}')

In [ ]:
# ─── Training Functions ───
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = device.type == 'cuda'

    pbar = tqdm(loader, desc='  Train', leave=False, unit='batch')
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds   = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds   = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / total, 100.0 * correct / total


# ─── Run Training ───
print(f'{'='*60}')
print(f'  🚀 Starting Training: {CONFIG["epochs"]} epochs on {device}')
print(f'{'='*60}')

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss,   val_acc   = validate_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    flag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch':       epoch,
            'model_type':  CONFIG['model_type'],
            'model_state': model.state_dict(),
            'val_acc':     val_acc,
            'class_names': CLASS_NAMES,
        }, best_ckpt)
        flag = ' ✅ BEST'

    elapsed = time.time() - t0
    print(
        f'Epoch [{epoch:>2}/{CONFIG["epochs"]}]  '
        f'Train: loss={train_loss:.4f} acc={train_acc:.1f}%  |  '
        f'Val: loss={val_loss:.4f} acc={val_acc:.1f}%  '
        f'({elapsed:.1f}s){flag}'
    )

print(f'\n🏁 Training Complete!  Best Val Accuracy: {best_val_acc:.2f}%')
print(f'   Checkpoint saved → {best_ckpt}')

# Save history
history_path = os.path.join(CONFIG['output_dir'], 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'   History saved → {history_path}')

## 📊 Section 7: Training Curves

In [ ]:
# ─── Plot Training Curves ───
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Cats vs Dogs — Training Results ({CONFIG["model_type"]})', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=4, linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   markersize=4, linewidth=2)
axes[0].fill_between(epochs_range, history['train_loss'], alpha=0.1, color='blue')
axes[0].fill_between(epochs_range, history['val_loss'],   alpha=0.1, color='red')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', markersize=4, linewidth=2)
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val Acc',   markersize=4, linewidth=2)
axes[1].fill_between(epochs_range, history['train_acc'], alpha=0.1, color='blue')
axes[1].fill_between(epochs_range, history['val_acc'],   alpha=0.1, color='red')
axes[1].axhline(y=90, color='green', linestyle='--', alpha=0.7, label='90% target')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CONFIG['output_dir'], 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Curves saved → {plot_path}')

## 🔍 Section 8: Evaluation & Confusion Matrix

In [ ]:
# ─── Evaluation on Validation Set ───
# Load best weights
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating'):
        images = images.to(device)
        outputs = model(images)
        preds   = (torch.sigmoid(outputs) >= 0.5).float().squeeze(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = [int(p) for p in all_preds]
all_labels = [int(l) for l in all_labels]

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

print('\n' + classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## 🐾 Section 9: Inference Demo

In [ ]:
# ─── Inference on Sample Images ───
infer_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

def predict_single(model, image, device):
    """Predict cat/dog from a PIL image."""
    model.eval()
    tensor = infer_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logit    = model(tensor)
        prob_dog = torch.sigmoid(logit).item()
        prob_cat = 1.0 - prob_dog
    label = 'Dog' if prob_dog >= 0.5 else 'Cat'
    conf  = prob_dog if prob_dog >= 0.5 else prob_cat
    return label, conf * 100, prob_cat * 100, prob_dog * 100


# Pick random val samples for demo
model.eval()
sample_count = 12
sample_indices = random.sample(val_idx.indices, sample_count)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('🐾 Inference Demo — Cats vs Dogs', fontsize=14, fontweight='bold')

for i, (ax, idx) in enumerate(zip(axes.flat, sample_indices)):
    raw_image, true_label = full_val_ds.inner[idx]
    true_cls = CLASS_NAMES[int(true_label) - 1]   # species: 1=Cat → 0, 2=Dog → 1

    pred_cls, conf, p_cat, p_dog = predict_single(model, raw_image, device)
    correct = pred_cls == true_cls

    ax.imshow(raw_image)
    color = '#2ecc71' if correct else '#e74c3c'
    icon  = '✓' if correct else '✗'
    ax.set_title(
        f"{icon} {pred_cls} ({conf:.0f}%)\nActual: {true_cls}",
        fontsize=8.5, fontweight='bold', color=color,
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'inference_demo.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Inference demo complete.')

## 🖼️ Section 9b: Custom Image Prediction — Upload Your Own Photo

Upload **any** JPEG or PNG from your local machine.  
The cell will run inference and show:
- **Left** — your uploaded image  
- **Right** — a horizontal confidence bar chart (green = predicted class, red = other)

> No retraining needed — the best checkpoint loaded above is reused.

In [ ]:
# ─── Custom Image Upload & Prediction ────────────────────────────────────────
# Upload a JPEG or PNG from your local machine, run the trained model,
# and display the image next to a colour-coded confidence bar chart.
# ─────────────────────────────────────────────────────────────────────────────

import io
from google.colab import files as colab_files

print("📁 Click 'Choose Files' below to upload a cat or dog image (JPEG / PNG):")
uploaded = colab_files.upload()

if not uploaded:
    print("⚠️  No file uploaded. Re-run this cell and select a file.")
else:
    # ── Load the uploaded image from raw bytes ─────────────────────────────
    filename   = list(uploaded.keys())[0]
    img_bytes  = uploaded[filename]
    upload_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # ── Run inference (reuses the model + best weights from Section 8) ─────
    pred_cls, conf, p_cat, p_dog = predict_single(model, upload_img, device)

    print(f"\n{'='*45}")
    print(f"  Image  : {filename}")
    print(f"  Result : {pred_cls}  ({conf:.1f}% confident)")
    print(f"  P(Cat) : {p_cat:.1f}%")
    print(f"  P(Dog) : {p_dog:.1f}%")
    print(f"{'='*45}")

    # ── Build side-by-side figure ──────────────────────────────────────────
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(
        f"Custom Prediction — {filename}",
        fontsize=13, fontweight="bold",
    )

    # Left: uploaded image (no normalisation — display as-is)
    ax_img.imshow(upload_img)
    ax_img.set_title("Uploaded Image", fontsize=10)
    ax_img.axis("off")

    # Right: horizontal confidence bar chart
    bar_labels = ["Cat", "Dog"]
    bar_values = [p_cat, p_dog]
    bar_colors = ["#2ecc71" if lbl == pred_cls else "#e74c3c" for lbl in bar_labels]

    bars = ax_bar.barh(bar_labels, bar_values, color=bar_colors, height=0.4)
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel("Confidence (%)", fontsize=10)
    ax_bar.set_title(
        f"Predicted: {pred_cls}  ({conf:.1f}%)",
        fontsize=11, fontweight="bold",
    )
    ax_bar.axvline(x=50, color="gray", linestyle="--", alpha=0.5, linewidth=1)

    for bar, val in zip(bars, bar_values):
        ax_bar.text(
            min(val + 1.5, 92),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%",
            va="center", ha="left", fontsize=11, fontweight="bold",
        )

    plt.tight_layout()

    # Save chart to output folder
    stem       = os.path.splitext(filename)[0]
    chart_path = os.path.join(CONFIG["output_dir"], f"prediction_{stem}.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Chart saved → {chart_path}")

## 💾 Section 10: Download Results

In [ ]:
# ─── Download model and plots from Colab ───
from google.colab import files
import zipfile

# Zip outputs
zip_path = '/content/catdog_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in ['best_model.pth', 'training_curves.png', 'confusion_matrix.png',
                  'inference_demo.png', 'training_history.json']:
        fpath = os.path.join(CONFIG['output_dir'], fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)

print(f'📦 Results zipped → {zip_path}')
print('⬇️  Downloading to your local machine...')
files.download(zip_path)
print('✅ Download complete!')

---
## ✅ Summary

| Step | Status |
|---|---|
| GPU Setup | ✅ |
| Oxford Pets Dataset | ✅ Auto-downloaded |
| ResNet-18 Transfer Learning | ✅ |
| Training with AMP | ✅ |
| Best Model Saved | ✅ `best_model.pth` |
| Training Curves | ✅ |
| Confusion Matrix | ✅ |
| Inference Demo | ✅ |
| Results Downloaded | ✅ |

**Backend Developer Agent** | SOLID Architecture | PyTorch | Educational ML